# PDF Question Answering with RAG
## Code Refactoring, Retrieval Evaluation, and Hallucination Control

**Author:** Layan Mousa

### Project objective
This notebook builds a reproducible Retrieval-Augmented Generation system for answering questions from the ISO/IEC 27001 PDF. It preserves source-page metadata, uses word-boundary chunking, applies one consistent normalized embedding path, controls context length before tokenization, adds confidence-based fallback behavior, and evaluates retrieval and answer generation separately.

### Grounding principle
The goal is not merely to generate an answer. Every accepted answer should be grounded in evidence retrieved from the provided PDF.

# 1. Install Dependencies
Installation is placed before imports, as required. Run this cell once in a fresh environment.

In [1]:
%pip install -q pypdf sentence-transformers chromadb transformers==4.55.4 sentencepiece accelerate pandas torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Import Libraries
Duplicate imports and repeated model-loading imports were removed.

In [2]:
from pathlib import Path
import hashlib
import re

import chromadb
import numpy as np
import pandas as pd
import torch

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

pd.set_option("display.max_colwidth", 180)

# 3. Project Configuration
All reusable settings are stored in one place. `MAX_DISTANCE` is an experimental confidence threshold and should be reviewed using the ten-question evaluation.

In [3]:
PDF_PATH = Path("iso27001.pdf")

COLLECTION_NAME = "iso27001_chunks"
CHROMA_PATH = "chroma_db"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL_NAME = "google/flan-t5-base"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100
TOP_K = 3

MAX_DISTANCE = 0.55
MAX_CONTEXT_CHARACTERS = 1800
MAX_NEW_TOKENS = 120

FALLBACK_MESSAGE = (
    "The information is not available in the provided document."
)

print("PDF:", PDF_PATH)
print("Chunk size:", CHUNK_SIZE, "words")
print("Chunk overlap:", CHUNK_OVERLAP, "words")
print("Top-k:", TOP_K)
print("Maximum accepted distance:", MAX_DISTANCE)

PDF: iso27001.pdf
Chunk size: 700 words
Chunk overlap: 100 words
Top-k: 3
Maximum accepted distance: 0.55


# 4. Load and Validate the PDF
The file is validated before reading. The actual filename is used later in chunk metadata.

In [4]:
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"PDF not found: {PDF_PATH}. "
        "Place iso27001.pdf in the same folder as this notebook."
    )

reader = PdfReader(str(PDF_PATH))

print("Document:", PDF_PATH.name)
print("Number of pages:", len(reader.pages))

Document: iso27001.pdf
Number of pages: 26


# 5. Extract Page-Aware Text
Each page is stored separately so retrieved evidence can report its true source page.

In [5]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text() or ""
    page_text = " ".join(page_text.split())

    if page_text.strip():
        pages.append({
            "page_number": page_number,
            "text": page_text,
        })

print("Pages with extracted text:", len(pages))
print(
    "Extracted characters:",
    sum(len(page["text"]) for page in pages),
)

Pages with extracted text: 26
Extracted characters: 56618


# 6. Extraction Quality Checks
Empty and unusually short pages are reported. Review the sample for headings, paragraphs, tables, symbols, and broken reading order.

**OCR note:** OCR may be needed only when pages are scanned images or extracted text is empty/garbled. This notebook does not apply OCR automatically.

In [6]:
empty_page_count = len(reader.pages) - len(pages)

short_pages = [
    page["page_number"]
    for page in pages
    if len(page["text"]) < 100
]

print("Empty pages:", empty_page_count)
print("Short pages:", short_pages)

if pages:
    print("\nFirst extracted page sample:\n")
    print(pages[0]["text"][:1200])

Empty pages: 0
Short pages: [6]

First extracted page sample:

Information security, cybersecurity and privacy protection — Information security management systems — Requirements Sécurité de l'information, cybersécurité et protection de la vie privée — Systèmes de management de la sécurité de l'information — Exigences INTERNATIONAL STANDARD ISO/IEC 27001 Third edition 2022-10 Reference number ISO/IEC 27001:2022(E) © ISO/IEC 2022 --``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---


In [7]:
extraction_observations = {
    "pages_total": len(reader.pages),
    "pages_with_text": len(pages),
    "empty_pages": empty_page_count,
    "short_pages": short_pages,
    "ocr_may_be_required": bool(empty_page_count > 0),
}

display(pd.DataFrame([extraction_observations]))

,pages_total,pages_with_text,empty_pages,short_pages,ocr_may_be_required
0,26,26,0,[6],False


# 7. Safe Word-Boundary Chunking
Chunk size and overlap are measured in words, preventing arbitrary cuts in the middle of words. Page boundaries are preserved.

In [8]:
def split_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive.")

    if chunk_overlap < 0:
        raise ValueError("chunk_overlap cannot be negative.")

    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size.")

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = max(end - chunk_overlap, start + 1)

    return chunks

**Why this overlap?**  
A 100-word overlap keeps nearby definitions, clause headings, and supporting sentences together across adjacent 700-word chunks. The configuration experiment later compares it with a smaller 400/50 setting.

# 8. Create Page-Aware Chunk Records
Every chunk stores the real PDF filename, page number, and page-local chunk index.

In [9]:
def create_chunk_records(
    page_records: list[dict],
    chunk_size: int,
    chunk_overlap: int,
) -> list[dict]:
    records = []

    for page in page_records:
        page_chunks = split_text(
            page["text"],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        for chunk_index, chunk_text in enumerate(page_chunks):
            chunk_id = (
                f"page_{page['page_number']}"
                f"_chunk_{chunk_index}"
            )

            records.append({
                "id": chunk_id,
                "text": chunk_text,
                "metadata": {
                    "source": PDF_PATH.name,
                    "page_number": page["page_number"],
                    "chunk_index": chunk_index,
                },
            })

    return records


chunk_records = create_chunk_records(
    pages,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunk_texts = [
    record["text"]
    for record in chunk_records
]

print("Total chunks:", len(chunk_records))
display(pd.DataFrame(chunk_records[:3]))

Total chunks: 26


,id,text,metadata
0,page_1_chunk_0,"Information security, cybersecurity and privacy protection — Information security management systems — Requirements Sécurité de l'information, cybersécurité et protection de la...","{'source': 'iso27001.pdf', 'page_number': 1, 'chunk_index': 0}"
1,page_2_chunk_0,"ii ISO/IEC 27001:2022(E) COPYRIGHT PROTECTED DOCUMENT © ISO/IEC 2022 All rights reserved. Unless otherwise specified, or required in the context of its implementation, no part ...","{'source': 'iso27001.pdf', 'page_number': 2, 'chunk_index': 0}"
2,page_3_chunk_0,ISO/IEC 27001:2022(E) Foreword ....................................................................................................................................................,"{'source': 'iso27001.pdf', 'page_number': 3, 'chunk_index': 0}"


# 9. Generate Normalized Embeddings Once
The same Sentence Transformer model is used for both document chunks and questions. Normalization makes cosine-distance retrieval consistent.

In [10]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("Embedding shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (26, 384)


# 10. Store Chunks and Metadata in ChromaDB
The collection is opened with `get_or_create_collection`. It is not silently deleted. `upsert` safely refreshes matching chunk IDs without broad exception suppression.

In [11]:
chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

collection.upsert(
    ids=[
        record["id"]
        for record in chunk_records
    ],
    documents=chunk_texts,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[
        record["metadata"]
        for record in chunk_records
    ],
)

print("Stored chunks:", collection.count())

Stored chunks: 26


# 11. One Consistent Retrieval Function
Questions are embedded externally with the same normalized embedding model. The inconsistent `query_texts=[question]` path was removed.

In [12]:
def retrieve_context(
    question: str,
    top_k: int = TOP_K,
) -> list[dict]:
    if not question or not question.strip():
        raise ValueError("Question cannot be empty.")

    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    results = collection.query(
        query_embeddings=[
            question_embedding.tolist()
        ],
        n_results=top_k,
        include=[
            "documents",
            "distances",
            "metadatas",
        ],
    )

    retrieved = []

    for text, distance, metadata in zip(
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0],
    ):
        retrieved.append({
            "text": text,
            "distance": float(distance),
            "metadata": metadata,
        })

    return retrieved

# 12. Review Retrieved Evidence
Page numbers and distances are displayed before generation.

In [13]:
sample_question = "What does ISO/IEC 27001 specify?"
sample_retrieved_items = retrieve_context(sample_question)

print("Question:", sample_question)

for rank, item in enumerate(
    sample_retrieved_items,
    start=1,
):
    print("\nRank:", rank)
    print("Source:", item["metadata"]["source"])
    print("Page:", item["metadata"]["page_number"])
    print("Distance:", round(item["distance"], 4))
    print(item["text"][:500])
    print("-" * 80)

Question: What does ISO/IEC 27001 specify?

Rank: 1
Source: iso27001.pdf
Page: 4
Distance: 0.2776
ISO/IEC 27001:2022(E) Foreword ISO (the International Organization for Standardization) and IEC (the International Electrotechnical Commission) form the specialized system for worldwide standardization. National bodies that are members of ISO or IEC participate in the development of International Standards through technical committees established by the respective organization to deal with particular fields of technical activity. ISO and IEC technical committees collaborate in fields of mutual i
--------------------------------------------------------------------------------

Rank: 2
Source: iso27001.pdf
Page: 7
Distance: 0.288
INTERNATIONAL ST ANDARD ISO/IEC 27001:2022(E) Information security, cybersecurity and privacy protection — Information security management systems — Requirements 1 Scope This document specifies the requirements for establishing, implementing, maintaining and continu

# 13. Confidence Threshold
Weak retrieval triggers the exact fallback message instead of allowing unsupported generation.

`0.55` is an initial experimental threshold, not a universal constant. Its effect is reviewed across answerable and unavailable questions.

In [14]:
def has_reliable_context(
    retrieved_items: list[dict],
    max_distance: float = MAX_DISTANCE,
) -> bool:
    if not retrieved_items:
        return False

    return (
        retrieved_items[0]["distance"]
        <= max_distance
    )

# 14. Context Budgeting Before Tokenization
Only complete retrieved chunks that fit within the character budget are selected. This avoids silently truncating a long prompt at `max_length=512`.

In [15]:
def build_context(
    retrieved_items: list[dict],
    max_characters: int = MAX_CONTEXT_CHARACTERS,
) -> str:
    selected_parts = []
    current_length = 0

    for item in retrieved_items:
        part = (
            f"[Source: {item['metadata']['source']}; "
            f"Page: {item['metadata']['page_number']}]\n"
            f"{item['text']}"
        )

        additional_length = len(part)
        if selected_parts:
            additional_length += 2

        if (
            current_length + additional_length
            > max_characters
        ):
            break

        selected_parts.append(part)
        current_length += additional_length

    return "\n\n".join(selected_parts)

# 15. Load FLAN-T5 Once
The tokenizer and generation model are loaded only once.

In [16]:
tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL_NAME
)

llm_model = (
    AutoModelForSeq2SeqLM
    .from_pretrained(
        GENERATION_MODEL_NAME
    )
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

llm_model = llm_model.to(device)
llm_model.eval()

print("Device:", device)

Device: cpu


# 16. One Grounded Answer Function
Retrieval, confidence checking, context selection, generation, and source reporting are combined in one function.

In [17]:
def generate_from_retrieved_items(
    question: str,
    retrieved_items: list[dict],
    max_distance: float = MAX_DISTANCE,
) -> dict:
    if not has_reliable_context(
        retrieved_items,
        max_distance=max_distance,
    ):
        return {
            "answer": FALLBACK_MESSAGE,
            "sources": [],
            "retrieved_items": retrieved_items,
            "context": "",
        }

    context = build_context(retrieved_items)

    if not context:
        return {
            "answer": FALLBACK_MESSAGE,
            "sources": [],
            "retrieved_items": retrieved_items,
            "context": "",
        }

    prompt = (
        "Answer the question using only the provided context.\n\n"
        "Rules:\n"
        "1. Do not use outside knowledge.\n"
        "2. Do not invent or infer unsupported information.\n"
        f"3. If unavailable, reply exactly: {FALLBACK_MESSAGE}\n"
        "4. Give a concise answer.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False,
    ).to(device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    ).strip()

    if not answer:
        answer = FALLBACK_MESSAGE

    sources = sorted({
        int(item["metadata"]["page_number"])
        for item in retrieved_items
    })

    return {
        "answer": answer,
        "sources": sources,
        "retrieved_items": retrieved_items,
        "context": context,
    }


def answer_question(
    question: str,
    top_k: int = TOP_K,
    max_distance: float = MAX_DISTANCE,
) -> dict:
    retrieved_items = retrieve_context(
        question,
        top_k=top_k,
    )

    return generate_from_retrieved_items(
        question,
        retrieved_items,
        max_distance=max_distance,
    )

# 17. Ten-Question Evaluation Dataset
Each question includes expected availability and keywords. The set contains five available, three detailed, and two unavailable questions.

In [18]:
test_questions = [
    {
        "category": "Available",
        "question": "What does ISO/IEC 27001 specify?",
        "answer_available": True,
        "expected_keywords": [
            "requirements",
            "information security",
            "management system",
        ],
    },
    {
        "category": "Available",
        "question": "What is the title of Clause 4?",
        "answer_available": True,
        "expected_keywords": [
            "context",
            "organization",
        ],
    },
    {
        "category": "Available",
        "question": "What is the title of Clause 5?",
        "answer_available": True,
        "expected_keywords": [
            "leadership",
        ],
    },
    {
        "category": "Available",
        "question": "What is the title of Clause 10?",
        "answer_available": True,
        "expected_keywords": [
            "improvement",
        ],
    },
    {
        "category": "Available",
        "question": "What document contains the terms and definitions?",
        "answer_available": True,
        "expected_keywords": [
            "ISO/IEC 27000",
        ],
    },
    {
        "category": "Detailed",
        "question": (
            "Summarize the purpose of the information "
            "security management system."
        ),
        "answer_available": True,
        "expected_keywords": [
            "confidentiality",
            "integrity",
            "availability",
        ],
    },
    {
        "category": "Detailed",
        "question": "Explain the purpose of internal audits.",
        "answer_available": True,
        "expected_keywords": [
            "conforms",
            "requirements",
        ],
    },
    {
        "category": "Detailed",
        "question": (
            "What should an organization do when "
            "a nonconformity occurs?"
        ),
        "answer_available": True,
        "expected_keywords": [
            "react",
            "corrective action",
        ],
    },
    {
        "category": "Unavailable",
        "question": "Who created ISO/IEC 27001?",
        "answer_available": False,
        "expected_keywords": [],
    },
    {
        "category": "Unavailable",
        "question": (
            "What is the certification cost "
            "of ISO/IEC 27001?"
        ),
        "answer_available": False,
        "expected_keywords": [],
    },
]

display(pd.DataFrame(test_questions))

,category,question,answer_available,expected_keywords
0,Available,What does ISO/IEC 27001 specify?,True,"[requirements, information security, management system]"
1,Available,What is the title of Clause 4?,True,"[context, organization]"
2,Available,What is the title of Clause 5?,True,[leadership]
3,Available,What is the title of Clause 10?,True,[improvement]
4,Available,What document contains the terms and definitions?,True,[ISO/IEC 27000]
5,Detailed,Summarize the purpose of the information security management system.,True,"[confidentiality, integrity, availability]"
6,Detailed,Explain the purpose of internal audits.,True,"[conforms, requirements]"
7,Detailed,What should an organization do when a nonconformity occurs?,True,"[react, corrective action]"
8,Unavailable,Who created ISO/IEC 27001?,False,[]
9,Unavailable,What is the certification cost of ISO/IEC 27001?,False,[]


# 18. Structured Retrieval and Answer Evaluation
The table records retrieval distance, source pages, fallback behavior, keyword-based answer correctness, and the generated answer.

In [19]:
evaluation_rows = []
retrieval_review_rows = []

for index, item in enumerate(
    test_questions,
    start=1,
):
    result = answer_question(
        item["question"]
    )

    answer_lower = result["answer"].lower()

    fallback_used = (
        result["answer"]
        == FALLBACK_MESSAGE
    )

    keyword_match = all(
        keyword.lower() in answer_lower
        for keyword in item[
            "expected_keywords"
        ]
    )

    answer_correct = (
        keyword_match
        if item["answer_available"]
        else fallback_used
    )

    best_distance = None
    if result["retrieved_items"]:
        best_distance = result[
            "retrieved_items"
        ][0]["distance"]

    source_pages = result["sources"]

    evaluation_rows.append({
        "Test": index,
        "Category": item["category"],
        "Question": item["question"],
        "Expected Available": item["answer_available"],
        "Best Distance": best_distance,
        "Fallback Used": fallback_used,
        "Keyword Match": keyword_match,
        "Answer Correct": answer_correct,
        "Source Pages": source_pages,
        "Answer": result["answer"],
    })

    retrieval_review_rows.append({
        "Test": index,
        "Question": item["question"],
        "Best Distance": best_distance,
        "Retrieved Pages": [
            retrieved["metadata"]["page_number"]
            for retrieved in result["retrieved_items"]
        ],
        "Retrieved Correct Context": "",
        "Reviewer Notes": "",
    })

evaluation_df = pd.DataFrame(
    evaluation_rows
)

retrieval_evaluation_df = pd.DataFrame(
    retrieval_review_rows
)

display(evaluation_df)

,Test,Category,Question,Expected Available,Best Distance,Fallback Used,Keyword Match,Answer Correct,Source Pages,Answer
0,1,Available,What does ISO/IEC 27001 specify?,True,0.277606,True,False,False,[],The information is not available in the provided document.
1,2,Available,What is the title of Clause 4?,True,0.802701,True,False,False,[],The information is not available in the provided document.
2,3,Available,What is the title of Clause 5?,True,0.782327,True,False,False,[],The information is not available in the provided document.
3,4,Available,What is the title of Clause 10?,True,0.837294,True,False,False,[],The information is not available in the provided document.
4,5,Available,What document contains the terms and definitions?,True,0.558932,True,False,False,[],The information is not available in the provided document.
5,6,Detailed,Summarize the purpose of the information security management system.,True,0.289047,True,False,False,[],The information is not available in the provided document.
6,7,Detailed,Explain the purpose of internal audits.,True,0.403546,True,False,False,[],The information is not available in the provided document.
7,8,Detailed,What should an organization do when a nonconformity occurs?,True,0.484631,False,False,False,"[15, 16, 19]",take action to control and correct it
8,9,Unavailable,Who created ISO/IEC 27001?,False,0.336869,True,True,True,[],The information is not available in the provided document.
9,10,Unavailable,What is the certification cost of ISO/IEC 27001?,False,0.360958,False,True,False,"[1, 4, 26]",03.100.70


# 19. Evaluate Retrieval Separately
Generation accuracy alone cannot show whether failure came from retrieval or from FLAN-T5. Inspect each retrieved passage and manually complete:

- `Retrieved Correct Context`: Yes / No
- `Reviewer Notes`: why the evidence was correct, incomplete, or irrelevant

In [20]:
display(retrieval_evaluation_df)

,Test,Question,Best Distance,Retrieved Pages,Retrieved Correct Context,Reviewer Notes
0,1,What does ISO/IEC 27001 specify?,0.277606,"[4, 7, 26]",,
1,2,What is the title of Clause 4?,0.802701,"[8, 7, 6]",,
2,3,What is the title of Clause 5?,0.782327,"[8, 9, 7]",,
3,4,What is the title of Clause 10?,0.837294,"[7, 8, 6]",,
4,5,What document contains the terms and definitions?,0.558932,"[7, 13, 4]",,
5,6,Summarize the purpose of the information security management system.,0.289047,"[5, 8, 9]",,
6,7,Explain the purpose of internal audits.,0.403546,"[15, 24, 10]",,
7,8,What should an organization do when a nonconformity occurs?,0.484631,"[16, 19, 15]",,
8,9,Who created ISO/IEC 27001?,0.336869,"[4, 26, 1]",,
9,10,What is the certification cost of ISO/IEC 27001?,0.360958,"[26, 4, 1]",,


# 20. Final Answer and Fallback Metrics
Answerable and unavailable questions are scored separately.

In [21]:
overall_accuracy = (
    evaluation_df["Answer Correct"].mean()
)

available_accuracy = (
    evaluation_df.loc[
        evaluation_df[
            "Expected Available"
        ] == True,
        "Answer Correct",
    ].mean()
)

fallback_accuracy = (
    evaluation_df.loc[
        evaluation_df[
            "Expected Available"
        ] == False,
        "Answer Correct",
    ].mean()
)

print(
    "Overall answer accuracy:",
    round(overall_accuracy, 3),
)
print(
    "Answerable-question accuracy:",
    round(available_accuracy, 3),
)
print(
    "Unavailable-question fallback accuracy:",
    round(fallback_accuracy, 3),
)

Overall answer accuracy: 0.1
Answerable-question accuracy: 0.0
Unavailable-question fallback accuracy: 0.5


# 21. Confidence-Threshold Review
The table below shows whether each question would be accepted or rejected at several candidate thresholds. Use the ten cases and manual evidence review to choose a threshold that balances valid answers against hallucination control.

In [22]:
candidate_thresholds = [
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
]

threshold_rows = []

for threshold in candidate_thresholds:
    accepted = (
        evaluation_df["Best Distance"]
        <= threshold
    )

    expected_available = (
        evaluation_df["Expected Available"]
    )

    threshold_rows.append({
        "Threshold": threshold,
        "Answerable Accepted Rate": (
            accepted[expected_available].mean()
        ),
        "Unavailable Rejected Rate": (
            (~accepted[~expected_available]).mean()
        ),
        "All Questions Accepted Rate": accepted.mean(),
    })

threshold_review_df = pd.DataFrame(
    threshold_rows
)

display(threshold_review_df)

,Threshold,Answerable Accepted Rate,Unavailable Rejected Rate,All Questions Accepted Rate
0,0.45,0.375,0.0,0.5
1,0.50,0.500,0.0,0.6
2,0.55,0.500,0.0,0.6
3,0.60,0.625,0.0,0.7
4,0.65,0.625,0.0,0.7


# 22. Compare Two RAG Configurations
The required experiment compares:

1. 400-word chunks, 50-word overlap, top-k 3  
2. 700-word chunks, 100-word overlap, top-k 3

For each setting, retrieval uses the same normalized embedding model. The experiment reports answer accuracy, fallback accuracy, and a manual retrieval-accuracy field.

In [23]:
def build_in_memory_index(
    page_records: list[dict],
    chunk_size: int,
    chunk_overlap: int,
) -> tuple[list[dict], np.ndarray]:
    records = create_chunk_records(
        page_records,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    texts = [
        record["text"]
        for record in records
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    return records, embeddings


def retrieve_from_in_memory_index(
    question: str,
    records: list[dict],
    embeddings: np.ndarray,
    top_k: int,
) -> list[dict]:
    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    cosine_similarities = (
        embeddings
        @ question_embedding
    )

    distances = 1 - cosine_similarities

    top_indices = np.argsort(
        distances
    )[:top_k]

    return [
        {
            "text": records[index]["text"],
            "distance": float(
                distances[index]
            ),
            "metadata": records[index]["metadata"],
        }
        for index in top_indices
    ]

In [24]:
experiment_configurations = [
    {
        "Chunk Size": 400,
        "Overlap": 50,
        "Top K": 3,
    },
    {
        "Chunk Size": 700,
        "Overlap": 100,
        "Top K": 3,
    },
]

experiment_rows = []

for config in experiment_configurations:
    experiment_records, experiment_embeddings = (
        build_in_memory_index(
            pages,
            chunk_size=config["Chunk Size"],
            chunk_overlap=config["Overlap"],
        )
    )

    configuration_correctness = []
    configuration_fallback_correctness = []

    for item in test_questions:
        retrieved_items = (
            retrieve_from_in_memory_index(
                item["question"],
                records=experiment_records,
                embeddings=experiment_embeddings,
                top_k=config["Top K"],
            )
        )

        result = generate_from_retrieved_items(
            item["question"],
            retrieved_items,
            max_distance=MAX_DISTANCE,
        )

        answer_lower = result["answer"].lower()
        fallback_used = (
            result["answer"]
            == FALLBACK_MESSAGE
        )

        keyword_match = all(
            keyword.lower() in answer_lower
            for keyword in item[
                "expected_keywords"
            ]
        )

        correct = (
            keyword_match
            if item["answer_available"]
            else fallback_used
        )

        configuration_correctness.append(
            correct
        )

        if not item["answer_available"]:
            configuration_fallback_correctness.append(
                fallback_used
            )

    experiment_rows.append({
        **config,
        "Chunks": len(experiment_records),
        "Retrieval Accuracy (Manual)": "",
        "Answer Accuracy": float(
            np.mean(configuration_correctness)
        ),
        "Fallback Accuracy": float(
            np.mean(
                configuration_fallback_correctness
            )
        ),
        "Reviewer Notes": "",
    })

experiment_results = pd.DataFrame(
    experiment_rows
)

display(experiment_results)

Token indices sequence length is longer than the specified maximum sequence length for this model (576 > 512). Running this sequence through the model will result in indexing errors


,Chunk Size,Overlap,Top K,Chunks,Retrieval Accuracy (Manual),Answer Accuracy,Fallback Accuracy,Reviewer Notes
0,400,50,3,27,,0.1,0.5,
1,700,100,3,26,,0.1,0.5,


**Experiment interpretation:**  
Select the configuration that retrieves correct evidence most consistently while maintaining strong answer and fallback accuracy. A smaller chunk may improve precision but lose surrounding context; a larger chunk may preserve context but introduce irrelevant text. Manual retrieval review remains necessary.

# 23. Concise Final System Summary
The duplicate final sample output from the original notebook was removed.

In [25]:
print("Document loaded successfully.")
print("Document:", PDF_PATH.name)
print("Number of pages:", len(reader.pages))
print("Extracted pages:", len(pages))
print("Stored chunks:", collection.count())
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Generation model:", GENERATION_MODEL_NAME)
print("Chunk size / overlap:", CHUNK_SIZE, "/", CHUNK_OVERLAP)
print("Top-k:", TOP_K)
print("Confidence threshold:", MAX_DISTANCE)

Document loaded successfully.
Document: iso27001.pdf
Number of pages: 26
Extracted pages: 26
Stored chunks: 26
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Generation model: google/flan-t5-base
Chunk size / overlap: 700 / 100
Top-k: 3
Confidence threshold: 0.55


# 24. Final Conclusion
The following cell produces a result-based conclusion after the notebook is run. Add the most common retrieval and generation failures from the manual reviewer notes before submission.

In [26]:
best_experiment = (
    experiment_results
    .sort_values(
        [
            "Answer Accuracy",
            "Fallback Accuracy",
        ],
        ascending=False,
    )
    .iloc[0]
)

retrieval_failure_notes = (
    "Complete the manual retrieval-review table to identify "
    "wrong pages, incomplete clauses, or overly broad chunks."
)

generation_failure_notes = (
    "Review cases where correct evidence was retrieved but "
    "FLAN-T5 omitted expected keywords, copied irrelevant text, "
    "or produced an incomplete summary."
)

threshold_effect = (
    "The confidence threshold rejected weakly matched questions "
    "and therefore reduced unsupported generation; however, "
    "it may also reject some valid questions and must be tuned "
    "against the evaluation set."
)

conclusion = f'''
The RAG pipeline extracted {len(pages)} text-bearing pages from
{PDF_PATH.name}, created page-aware word-boundary chunks, generated normalized
embeddings with {EMBEDDING_MODEL_NAME}, and stored them in ChromaDB.

Selected main configuration:
- Chunk size: {CHUNK_SIZE} words
- Chunk overlap: {CHUNK_OVERLAP} words
- Top-k: {TOP_K}
- Confidence threshold: {MAX_DISTANCE}

Evaluation results:
- Overall answer accuracy: {overall_accuracy:.3f}
- Answerable-question accuracy: {available_accuracy:.3f}
- Unavailable-question fallback accuracy: {fallback_accuracy:.3f}

The strongest tested configuration by automated answer/fallback scores used
a chunk size of {int(best_experiment["Chunk Size"])} words, an overlap of
{int(best_experiment["Overlap"])} words, and top-k
{int(best_experiment["Top K"])}. Final selection should also consider the
manually reviewed retrieval accuracy.

Common retrieval failures:
{retrieval_failure_notes}

Common generation failures:
{generation_failure_notes}

Hallucination control:
{threshold_effect}

Limitations:
1. PDF text extraction can lose table layout and document structure.
2. FLAN-T5-base is a relatively small generation model and can produce
   incomplete or weakly synthesized answers.
3. The confidence threshold is based on a small ten-question test set.
4. Keyword scoring is only a lightweight proxy for semantic answer quality.
5. Manual evidence review is still required to separate retrieval errors from
   generation errors.

Future improvement:
Use structure-aware or semantic chunking with clause headings, then evaluate
retrieval with labelled relevant pages or chunks.
'''

print(conclusion)


The RAG pipeline extracted 26 text-bearing pages from
iso27001.pdf, created page-aware word-boundary chunks, generated normalized
embeddings with sentence-transformers/all-MiniLM-L6-v2, and stored them in ChromaDB.

Selected main configuration:
- Chunk size: 700 words
- Chunk overlap: 100 words
- Top-k: 3
- Confidence threshold: 0.55

Evaluation results:
- Overall answer accuracy: 0.100
- Answerable-question accuracy: 0.000
- Unavailable-question fallback accuracy: 0.500

The strongest tested configuration by automated answer/fallback scores used
a chunk size of 400 words, an overlap of
50 words, and top-k
3. Final selection should also consider the
manually reviewed retrieval accuracy.

Common retrieval failures:
Complete the manual retrieval-review table to identify wrong pages, incomplete clauses, or overly broad chunks.

Common generation failures:
Review cases where correct evidence was retrieved but FLAN-T5 omitted expected keywords, copied irrelevant text, or produced an incomp